In [1]:
import pandas as pd
import plotly.graph_objects as go

In [2]:
charges = pd.read_csv("data/Charges.csv")
subscriber_tariffs = pd.read_csv("data/Tariff_plans_change.csv")
suspended_subscribers = pd.read_csv("data/Suspended.csv")

Setting previous tariff to another column and removing wrong rows with different subscribers

In [3]:
subscriber_tariffs["previous_tariff"] = subscriber_tariffs["TARIFF_PLAN_ID"].shift(1)
mask = subscriber_tariffs["SUBSCRIBER_ID"] == subscriber_tariffs["SUBSCRIBER_ID"].shift(1)
changes_by_subscriber = subscriber_tariffs[mask]

Grouping by new and old tariffs

In [4]:
changes_counts = changes_by_subscriber.groupby(["previous_tariff", "TARIFF_PLAN_ID"])["SUBSCRIBER_ID"].count().to_dict()

In [5]:
source = []
target = []
values = []

for change_type, count in changes_counts.items():
    source.append(int(change_type[0]) - 1)
    target.append(change_type[1] + 4)
    values.append(count)

labels = ["Previous tariff 1", "Previous tariff 2", "Previous tariff 3", 
          "Previous tariff 4", "Previous tariff 5", "New tariff 1", 
          "New tariff 2", "New tariff 3", "New tariff 4", "New tariff 5"]

In [6]:
fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 10,
      thickness = 40,
      line = dict(color = "black", width = 0.5),
      label = labels,
      color = "blue"
    ),
    link = dict(
      arrowlen=15,
      source = source,
      target = target,
      value = values
  ))])

fig.update_layout(
    autosize=False,
    width=800,
    height=800,
    font_size=20,
    title = "Switching between tariffs"
)
fig.show()

Merging two dataframes by subscriber

In [9]:
tariffs_charges = changes_by_subscriber.merge(charges, left_on="SUBSCRIBER_ID", right_on="SUBSCRIBER_ID")
tariffs_charges["START_DTTM"] = pd.to_datetime(tariffs_charges["START_DTTM"])
tariffs_charges["BILL_MONTH"] = pd.to_datetime(tariffs_charges["BILL_MONTH"])

Calculating how many months have passed since the tariff change. And extracting data with less than 3 months.

In [10]:
tariffs_charges["time_from_change"] = tariffs_charges["BILL_MONTH"].dt.month - tariffs_charges["START_DTTM"].dt.month
three_month_after = tariffs_charges[(tariffs_charges["time_from_change"] > 0) & (tariffs_charges["time_from_change"] < 3)]
three_month_before = tariffs_charges[(tariffs_charges["time_from_change"] > -4) & (tariffs_charges["time_from_change"] < 0)]

Grouping by old and new tariff and calculating mean charge.

In [ ]:
after_charges_mean = three_month_after.groupby(["previous_tariff", "TARIFF_PLAN_ID"])["CHARGES"].mean().to_dict()
before_charges_mean = three_month_before.groupby(["previous_tariff", "TARIFF_PLAN_ID"])["CHARGES"].mean().to_dict()

In [13]:
before_means = {i: [] for i in range(1, 6)}
after_means = {i: [] for i in range(1, 6)}
labels = {i: [] for i in range(1, 6)}

for change_type, mean in before_charges_mean.items():
    before_means[change_type[0]].append(mean)
    after_means[change_type[0]].append(after_charges_mean[change_type])
    labels[change_type[0]].append(str(int(change_type[0])) + " to " + str(change_type[1]))

In [14]:
for i in range(1, 6):
    fig = go.Figure(data=[go.Bar(
            name = 'Before change',
            x = labels[i],
            y = before_means[i]
        ),
                            go.Bar(
            name = 'After change',
            x = labels[i],
            y = after_means[i]
        )
        ])

    fig.update_layout(yaxis_range=[0, 14])        
    fig.show()

Doing the same thing for subscribers suspends

In [15]:
tariffs_suspendings = changes_by_subscriber.merge(suspended_subscribers, left_on="SUBSCRIBER_ID", right_on="SUBSCRIBER_ID")
tariffs_suspendings["START_DTTM"] = pd.to_datetime(tariffs_suspendings["START_DTTM"])
tariffs_suspendings["BILL_MONTH"] = pd.to_datetime(tariffs_suspendings["START_DT"])

In [16]:
tariffs_suspendings["time_from_change"] = tariffs_suspendings["BILL_MONTH"].dt.month - tariffs_suspendings["START_DTTM"].dt.month
three_month_after = tariffs_suspendings[(tariffs_suspendings["time_from_change"] > 0) & (tariffs_suspendings["time_from_change"] < 3)]
three_month_before = tariffs_suspendings[(tariffs_suspendings["time_from_change"] > -4) & (tariffs_suspendings["time_from_change"] < 0)]

In [17]:
after_charges_mean = three_month_after.groupby(["previous_tariff", "TARIFF_PLAN_ID"])["STATUS"].count().to_dict()
before_charges_mean = three_month_before.groupby(["previous_tariff", "TARIFF_PLAN_ID"])["STATUS"].count().to_dict()

In [18]:
before_means = {i: [] for i in range(1, 6)}
after_means = {i: [] for i in range(1, 6)}
labels = {i: [] for i in range(1, 6)}

for change_type, mean in before_charges_mean.items():
    before_means[change_type[0]].append(mean)
    after_means[change_type[0]].append(after_charges_mean[change_type])
    labels[change_type[0]].append(str(int(change_type[0])) + " to " + str(change_type[1]))

In [19]:
for i in range(1, 6):
    fig = go.Figure(data=[go.Bar(
            name = 'Before change',
            x = labels[i],
            y = before_means[i]
        ),
                            go.Bar(
            name = 'After change',
            x = labels[i],
            y = after_means[i]
        )
        ])

    fig.update_layout(yaxis_range=[0, 200])        
    fig.show()